# WOSAC Baseline: Colab Smoke Test (01)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/wosac-baseline/notebooks/01_colab_smoke_test.ipynb)

Goal: run one reproducible end-to-end smoke pipeline for the baseline pack.

Success criteria:
- runtime bootstrap succeeds without manual edits,
- config and run contract validate,
- baseline workflow entrypoint executes,
- metrics artifact is written to repo and Drive.


In [1]:
# Step 1: Repo sync + deterministic runtime bootstrap
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for path in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config

runtime_cfg = wosac_colab_runtime_config(
    repo_url=REPO_URL,
    repo_branch="main",
    repo_dir=REPO_DIR,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)
print("drive_status:", bootstrap.drive_status.detail)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


[repo] existing checkout: /content/wosac-sim-agents-experiments
[drive] /content/drive already mounted
[drive] Verified read/write access: /content/drive/MyDrive/wosac_experiments
[setup] Starting deterministic environment bootstrap
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__, scipy.__version__, sklearn.__version__, jax.__version__)
[setup] Core runtime already healthy; skipping dependency install (ok 2.2.6 2.2.3 1.14.1 1.6.1 0.7.2).
[setup] $ /usr/bin/python3 -c import numpy as np; from numpy._core.umath import _center, _expandtabs; print(np.__version__, np.__file__)
[setup] NumPy probe passed (2.2.6 /usr/local/lib/python3.12/dist-packages/numpy/__init__.py).
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__

In [2]:
# Step 2: Load experiment config and compute config hash
import hashlib
import json

from src.workflows import bootstrap_experiment_pack, load_experiment_config

EXPERIMENT_SLUG = "wosac-baseline"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")

cfg_str = json.dumps(cfg, sort_keys=True, indent=2)
cfg_hash = hashlib.sha256(cfg_str.encode("utf-8")).hexdigest()

print("experiment_slug:", EXPERIMENT_SLUG)
print("cfg_hash:", cfg_hash)
print(cfg_str)


experiment_slug: wosac-baseline
cfg_hash: 957507a8a8ad2d674e85680382eceb9c407568638e4d410b04d91dc9ad6788fc
{
  "evaluation": {
    "primary_metric": "realism_meta_metric",
    "report_metrics": [
      "kinematic_metrics",
      "interactive_metrics",
      "map_based_metrics",
      "min_ade",
      "simulated_collision_rate",
      "simulated_offroad_rate",
      "simulated_traffic_light_violation_rate"
    ]
  },
  "objective": "Establish first reproducible benchmark score before novelty attempts.",
  "repo": {
    "branch": "main",
    "repo_dir": "/content/wosac-sim-agents-experiments",
    "url": "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
  },
  "run": {
    "n_rollouts_per_scenario": 32,
    "n_shards": 1,
    "notes": "Keep unchanged until first stable score is recorded.",
    "num_future_seconds": 8,
    "num_history_seconds": 1,
    "persist_root": "/content/drive/MyDrive/wosac_experiments",
    "resume_from_existing": true,
    "run_enabled": true,
  

In [3]:
# Step 3: Run context and fast-fail validation
from datetime import datetime, timezone
from pathlib import Path

RUN = dict(cfg.get("run", {}))
EVAL = dict(cfg.get("evaluation", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "wosac_baseline"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
N_SHARDS = int(RUN.get("n_shards", 1))
SHARD_ID = int(RUN.get("shard_id", 0))
RESUME_FROM_EXISTING = bool(RUN.get("resume_from_existing", True))
RUN_ENABLED = bool(RUN.get("run_enabled", True))

assert PERSIST_ROOT.strip(), "persist_root is required"
assert N_SHARDS >= 1, "n_shards must be >= 1"
assert 0 <= SHARD_ID < N_SHARDS, "shard_id must be in [0, n_shards)"

RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)

print("RUN_NAME:", RUN_NAME)
print("RUN_PREFIX:", RUN_PREFIX)
print("RUN_TAG:", RUN_TAG)
print("persist_run_dir:", persist_run_dir)
print("RESUME_FROM_EXISTING:", RESUME_FROM_EXISTING)
print("RUN_ENABLED:", RUN_ENABLED)


RUN_NAME: dev
RUN_PREFIX: wosac_baseline
RUN_TAG: 20260303T133136Z
persist_run_dir: /content/drive/MyDrive/wosac_experiments/wosac_baseline_dev
RESUME_FROM_EXISTING: True
RUN_ENABLED: True


In [4]:
# Step 4: Optional data-shard probe (GCS-first; set WOSAC_SAMPLE_TFRECORD to override)
from pathlib import Path
import subprocess
import tensorflow as tf


def _bool_env(name: str, default: bool = False) -> bool:
    text = os.environ.get(name, '').strip().lower()
    if text in {'1', 'true', 'yes', 'y', 'on'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'off'}:
        return False
    return bool(default)


def _active_gcloud_account() -> str:
    try:
        out = subprocess.check_output(
            [
                'bash',
                '-lc',
                "gcloud auth list --filter=status:ACTIVE --format='value(account)'",
            ],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        return out.splitlines()[0].strip() if out else ''
    except Exception:
        return ''


def _ensure_colab_gcs_auth_optional() -> None:
    if 'google.colab' not in sys.modules:
        return
    force_auth = _bool_env('WOSAC_FORCE_GCS_AUTH', False)
    if force_auth:
        subprocess.run(['bash', '-lc', 'gcloud auth revoke --all --quiet'], check=False)
    before = _active_gcloud_account()
    if (not before) or force_auth:
        try:
            from google.colab import auth  # type: ignore
            auth.authenticate_user()
        except Exception as exc:
            print(f'Colab auth skipped or failed: {exc}')
    print('Active gcloud account:', _active_gcloud_account() or '<none>')


def _dedupe_keep_order(values: list[str]) -> list[str]:
    out: list[str] = []
    for v in values:
        t = str(v).strip().rstrip('/')
        if t and t not in out:
            out.append(t)
    return out


def _discover_probe_tfrecord() -> tuple[str, str]:
    explicit = os.environ.get('WOSAC_SAMPLE_TFRECORD', '').strip()
    if explicit:
        return explicit, 'env_override'

    root_env = os.environ.get('WOSAC_GCS_DATASET_ROOT', '').strip()
    split_env = os.environ.get('WOSAC_GCS_SPLIT', 'validation').strip() or 'validation'
    roots = _dedupe_keep_order([
        root_env,
        'gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario',
        'gs://waymo_open_dataset_motion_v_1_3_1/scenario',
        'gs://waymo_open_dataset_motion_v_1_2_0/uncompressed/scenario',
        'gs://waymo_open_dataset_motion_v_1_2_0/scenario',
    ])
    splits = _dedupe_keep_order([
        split_env,
        'validation',
        'validation_interactive',
        'training',
        'training_20s',
        'testing',
        'testing_interactive',
        'test',
    ])

    templates = [
        '{root}/{split}/{split}.tfrecord-00000-of-*',
        '{root}/{split}/{split}.tfrecord-*',
        '{root}/{split}/*.tfrecord-*',
        '{root}/{split}/*.tfrecord',
    ]

    for root in roots:
        for split in splits:
            for templ in templates:
                pattern = templ.format(root=root, split=split)
                matches = sorted(tf.io.gfile.glob(pattern))
                if matches:
                    return matches[0], f'auto_discovered:{pattern}'
    return '', f'no_match_for_roots:{roots[:2]}'


def _path_exists(sample_path: str) -> bool:
    if not sample_path:
        return False
    if sample_path.startswith('gs://'):
        if '*' in sample_path:
            return bool(tf.io.gfile.glob(sample_path))
        return bool(tf.io.gfile.exists(sample_path) or tf.io.gfile.glob(sample_path))
    p = Path(sample_path).expanduser()
    return bool(p.exists() and p.is_file())


_ensure_colab_gcs_auth_optional()
sample_tfrecord, discover_detail = _discover_probe_tfrecord()
exists = _path_exists(sample_tfrecord)

data_probe = {
    'env_var': 'WOSAC_SAMPLE_TFRECORD',
    'path': sample_tfrecord,
    'exists': exists,
    'quick_probe_pass': bool(exists or not sample_tfrecord),
    'detail': discover_detail if sample_tfrecord else (
        'No shard resolved from GCS for optional probe; set WOSAC_SAMPLE_TFRECORD or WOSAC_GCS_DATASET_ROOT/WOSAC_GCS_SPLIT.'
    ),
}

print(data_probe)
quick_probe_pass = bool(data_probe['quick_probe_pass'])
assert quick_probe_pass, 'Quick probe failed. Set a valid WOSAC_SAMPLE_TFRECORD (gs://... or local path).'



{'env_var': 'WOSAC_SAMPLE_TFRECORD', 'path': '', 'exists': False, 'quick_probe_pass': True, 'detail': 'Set WOSAC_SAMPLE_TFRECORD to a TFRecord path for real data probe.'}


In [5]:
# Step 5: Run baseline workflow entrypoint (ingests evaluator outputs if provided)
from src.workflows import run_wosac_baseline_flow

OFFICIAL_METRICS_JSON = os.environ.get("WOSAC_OFFICIAL_METRICS_JSON", "").strip()
OFFICIAL_METRICS_CSV = os.environ.get("WOSAC_OFFICIAL_METRICS_CSV", "").strip()

flow_bundle = run_wosac_baseline_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    n_rollouts_per_scenario=int(RUN.get("n_rollouts_per_scenario", 32)),
    num_history_seconds=int(RUN.get("num_history_seconds", 1)),
    num_future_seconds=int(RUN.get("num_future_seconds", 8)),
    official_metrics_json=OFFICIAL_METRICS_JSON,
    metrics_csv=OFFICIAL_METRICS_CSV,
)

print("flow_summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))
print("flow_metrics:")
print(json.dumps(flow_bundle.metrics, indent=2, sort_keys=True))
print("flow_artifacts:")
print(json.dumps(flow_bundle.artifacts, indent=2, sort_keys=True))

metrics = dict(flow_bundle.metrics)
metrics_note = "official_metrics_loaded" if metrics.get("realism_meta_metric") is not None else "dry_run_no_official_metrics"


flow_summary:
{
  "config_hash": "7e4a6e266b1bf93451684ab9c1fef9f0bcefc50a24436bf1c5fb56293e6cf588",
  "created_utc": "2026-03-03T13:31:47Z",
  "kwargs": {
    "metrics_csv": "",
    "n_rollouts_per_scenario": 32,
    "num_future_seconds": 8,
    "num_history_seconds": 1,
    "official_metrics_json": "",
    "persist_root": "/content/drive/MyDrive/wosac_experiments",
    "repo_root": ".",
    "run_name": "dev",
    "run_prefix": "wosac_baseline",
    "run_tag": "20260303T133136Z"
  },
  "message": "No evaluator metrics found yet. Generated dry-run artifacts and metadata.",
  "metric_ingest_error": "",
  "metrics_source": "none",
  "n_rollouts_per_scenario": 32,
  "persist_root": "/content/drive/MyDrive/wosac_experiments",
  "repo_commit": "192812804ba0e2706e981746c0db9732fdc7ccea",
  "repo_root": "/content/wosac-sim-agents-experiments",
  "run_name": "dev",
  "run_prefix": "wosac_baseline",
  "run_tag": "20260303T133136Z",
  "status": "dry_run"
}
flow_metrics:
{
  "realism_meta_metric"

In [6]:
# Step 6: Write baseline metric artifact to Drive
import subprocess

from pathlib import Path

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_commit = "unknown"

payload = {
    "run_id": "baseline_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "primary_metric": str(EVAL.get("primary_metric", "realism_meta_metric")),
    "metrics": metrics,
    "flow_summary": flow_bundle.summary,
    "flow_artifacts": flow_bundle.artifacts,
    "data_probe": data_probe,
    "notes": [
        "Smoke test validates pipeline contract and artifact writing.",
        "Set WOSAC_OFFICIAL_METRICS_JSON or WOSAC_OFFICIAL_METRICS_CSV to ingest evaluator outputs.",
        f"flow_status={flow_bundle.summary.get('status')}",
        f"metrics_note={metrics_note}",
    ],
    "provenance": {
        "repo_commit": git_commit,
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

drive_metrics_path = persist_run_dir / "outputs" / "baseline_v0_metrics.json"
drive_metrics_path.parent.mkdir(parents=True, exist_ok=True)
drive_metrics_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

print("drive_metrics_path:", drive_metrics_path)
payload


drive_metrics_path: /content/drive/MyDrive/wosac_experiments/wosac_baseline_dev/outputs/baseline_v0_metrics.json


{'run_id': 'baseline_v0',
 'status': 'dry_run',
 'run_tag': '20260303T133136Z',
 'primary_metric': 'realism_meta_metric',
 'metrics': {'realism_meta_metric': None,
  'simulated_collision_rate': None,
  'simulated_offroad_rate': None,
  'simulated_traffic_light_violation_rate': None},
 'flow_summary': {'status': 'dry_run',
  'message': 'No evaluator metrics found yet. Generated dry-run artifacts and metadata.',
  'run_tag': '20260303T133136Z',
  'run_name': 'dev',
  'run_prefix': 'wosac_baseline',
  'persist_root': '/content/drive/MyDrive/wosac_experiments',
  'n_rollouts_per_scenario': 32,
  'metrics_source': 'none',
  'metric_ingest_error': '',
  'repo_root': '/content/wosac-sim-agents-experiments',
  'repo_commit': '192812804ba0e2706e981746c0db9732fdc7ccea',
  'created_utc': '2026-03-03T13:31:47Z',
  'config_hash': '7e4a6e266b1bf93451684ab9c1fef9f0bcefc50a24436bf1c5fb56293e6cf588',
  'kwargs': {'run_tag': '20260303T133136Z',
   'run_name': 'dev',
   'run_prefix': 'wosac_baseline',
  